# Model Tracker and Submission Manager
### Purpose: Track all models in checkpoint directories and manage submissions


In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

import numpy as np
import torch
from src.utils.config import Config
from src.utils.helpers import init_notebook, p, t, c
from src.data.checkpoint import generate_best_model_report, main_model_tracking_pipeline, show_model_details
import random

# Initialize configuration
config = Config.load(root = root)

init_notebook(config.train.seed)


In [ ]:
torch.manual_seed(config.train.seed)
np.random.seed(config.train.seed)
random.seed(config.train.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(config.train.seed)
    torch.cuda.manual_seed_all(config.train.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
p("Device", device)
p("Config loaded", config.paths.root)


### Execute Pipeline


In [ ]:
# Run the main pipeline
t("Executing Model Tracking and Submission Management Pipeline")

try:
    results = main_model_tracking_pipeline(config)
    p("Pipeline completed successfully!", color1 = c.GREEN)
except Exception as e:
    p(f"Pipeline failed with error: {str(e)}", color1 = c.RED)
    import traceback

    traceback.print_exc()


### Show details for the best model


In [ ]:
if 'results' in locals() and results:
    ranked_models = results.get("ranked_models", [])
    if ranked_models:
        t("Best Model Details")
        show_model_details(ranked_models[0])


### Generate Final Best Model Report


In [ ]:
if 'results' in locals() and results:
    ranked_models = results.get("ranked_models", [])
    if ranked_models:
        best_report_path = generate_best_model_report(ranked_models, config)
